# Fast exact matcher benchmarks

This notebook benchmarks a specialized exact matcher for two common score families:

1. **Equality:** a vertex pair scores the matched token weight if the labels are identical, else `0`.
2. **Any-overlap:** labels are token sets / multisets, and a vertex pair scores the maximum token weight over the intersection.

The key comparison points are:

- current generic exact DP (`align_trees_algorithm1`)
- specialized fast matcher with **pair-local encoding**
- specialized fast matcher with **dataset-level pre-encoding** (the best regime when matching many trees repeatedly)

The notebook also checks score/path agreement against the generic exact DP.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "path_matcher").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing pyproject.toml and src/path_matcher.")


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
for path in (str(REPO_ROOT), str(SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Repo root:", REPO_ROOT)


In [ ]:

import os
import sys
import time
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 0
random.seed(SEED)
np.random.seed(SEED)

USING_SYNTHETIC_FALLBACK = False

try:
    from path_matcher.fast_match import FastTreePathMatcher, HAVE_NUMBA
    from path_matcher.needleman_wunsch_tree import align_trees_algorithm1
    from path_matcher.tree_data import TreeData
    from path_matcher.igraph_io import igraph_to_treedata
    try:
        from path_matcher.planted_path_sampler import (
            ToyModelConfig,
            make_default_tree_samplers,
            sample_toy_model,
        )
        HAVE_SAMPLER = True
    except Exception:
        HAVE_SAMPLER = False
except Exception:
    # Loose-file fallback
    from fast_match import FastTreePathMatcher, HAVE_NUMBA
    from needleman_wunsch_tree import align_trees_algorithm1
    from tree_data import TreeData
    from igraph_io import igraph_to_treedata
    HAVE_SAMPLER = False

print("Numba available:", HAVE_NUMBA)
print("Sampler available:", HAVE_SAMPLER)


In [ ]:

# Configuration

# Keep defaults light enough for interactive work.
RUN_HEAVY = True

if RUN_HEAVY:
    N_TREES = 80
    TREE_SIZE = 180
    N_PAIRS = 120
    N_CLASSES = 4
    N_PER_CLASS = 20
else:
    N_TREES = 36
    TREE_SIZE = 120
    N_PAIRS = 60
    N_CLASSES = 3
    N_PER_CLASS = 12

ALPHABET_SIZE = 16
OVERLAP_LABEL_WIDTH = 3   # average width used in the overlap benchmark
OVERLAP_EXTRA_TOKENS = 2  # tokens added on top of the base singleton label

print(dict(
    RUN_HEAVY=RUN_HEAVY,
    N_TREES=N_TREES,
    TREE_SIZE=TREE_SIZE,
    N_PAIRS=N_PAIRS,
    N_CLASSES=N_CLASSES,
    N_PER_CLASS=N_PER_CLASS,
    ALPHABET_SIZE=ALPHABET_SIZE,
))


In [ ]:

def random_tree_parent(n: int) -> np.ndarray:
    parent = np.empty(n, dtype=np.int32)
    parent[0] = -1
    for i in range(1, n):
        parent[i] = random.randrange(i)
    return parent

def synthetic_treedata_dataset(n_trees: int, tree_size: int, alphabet_size: int):
    alphabet = [f"s{i}" for i in range(alphabet_size)]
    trees = []
    class_ids = []
    for i in range(n_trees):
        parent = random_tree_parent(tree_size)
        labels = [random.choice(alphabet) for _ in range(tree_size)]
        trees.append(TreeData(parent=parent, label=labels, orig_index=np.arange(tree_size, dtype=np.int32)))
        class_ids.append(i % max(1, N_CLASSES))
    return trees, class_ids, alphabet

def sampled_treedata_dataset(n_classes: int, n_per_class: int, tree_size: int, alphabet_size: int):
    cfg = ToyModelConfig(
        n_classes=n_classes,
        n_per_class=n_per_class,
        planted_seq_len=12,
        alphabet_size=alphabet_size,
        p_obs=1.0,
        dirichlet_alpha=1.0,
        seed=SEED,
    )
    samplers = make_default_tree_samplers(
        max_depth=10,
        lam=1.7,
        max_nodes=tree_size,
        min_nodes=max(10, int(0.75 * tree_size)),
        require_full_depth=True,
        max_tries=5000,
    )
    graphs, class_ids, _class_sequences = sample_toy_model(cfg, tree_path_sampler=samplers["gw_bfs"])
    trees = [igraph_to_treedata(G, phi_name="label", order="auto", strict_tree=True) for G in graphs]
    alphabet = sorted({lab for T in trees for lab in T.label})
    return trees, class_ids, alphabet

if HAVE_SAMPLER:
    td_eq_trees, class_ids, alphabet = sampled_treedata_dataset(
        n_classes=N_CLASSES,
        n_per_class=N_PER_CLASS,
        tree_size=TREE_SIZE,
        alphabet_size=ALPHABET_SIZE,
    )
else:
    USING_SYNTHETIC_FALLBACK = True
    td_eq_trees, class_ids, alphabet = synthetic_treedata_dataset(
        n_trees=N_TREES,
        tree_size=TREE_SIZE,
        alphabet_size=ALPHABET_SIZE,
    )

print("Synthetic fallback:", USING_SYNTHETIC_FALLBACK)
print("Number of trees:", len(td_eq_trees))
print("Tree size example:", td_eq_trees[0].n)
print("Alphabet size in data:", len(alphabet))


In [ ]:

def build_same_class_pairs(class_ids, n_pairs, seed=SEED):
    rng = random.Random(seed)
    by_class = defaultdict(list)
    for i, c in enumerate(class_ids):
        by_class[int(c)].append(i)
    pool = []
    for c, idx in by_class.items():
        for a in range(len(idx)):
            for b in range(a + 1, len(idx)):
                pool.append((idx[a], idx[b], c))
    rng.shuffle(pool)
    if not pool:
        raise RuntimeError("No within-class pairs available.")
    pool = pool[: min(n_pairs, len(pool))]
    return [(a, b) for (a, b, _c) in pool]

pairs = build_same_class_pairs(class_ids, N_PAIRS)
print("Number of benchmark pairs:", len(pairs))
print("First few pairs:", pairs[:5])


In [ ]:

def make_token_weights(alphabet):
    # Mildly heterogeneous weights to avoid degenerate all-1 ties everywhere.
    return {tok: float((i % 4) + 1) for i, tok in enumerate(alphabet)}

token_weights = make_token_weights(alphabet)

def w_eq(a, b, weights=token_weights):
    return float(weights[a]) if a == b else 0.0

def lift_to_overlap_tree(tree: TreeData, alphabet, extra_tokens=2, seed=SEED):
    rng = random.Random(seed + int(tree.n) + int(np.sum(tree.parent + 1)))
    new_labels = []
    alphabet_list = list(alphabet)
    for lab in tree.label:
        toks = {lab}
        # add a couple of distractors
        if extra_tokens > 0:
            k = rng.randint(0, extra_tokens)
            if k > 0:
                extras = rng.sample(alphabet_list, k=min(k, len(alphabet_list)))
                toks.update(extras)
        new_labels.append(tuple(sorted(toks)))
    return TreeData(parent=np.asarray(tree.parent, dtype=np.int32).copy(),
                    label=new_labels,
                    orig_index=np.asarray(tree.orig_index, dtype=np.int32).copy())

td_overlap_trees = [
    lift_to_overlap_tree(T, alphabet=alphabet, extra_tokens=OVERLAP_EXTRA_TOKENS, seed=SEED + i)
    for i, T in enumerate(td_eq_trees)
]

def w_overlap(a, b, weights=token_weights):
    common = set(a) & set(b)
    if not common:
        return 0.0
    return max(float(weights[t]) for t in common)

print("Example equality label:", td_eq_trees[0].label[0])
print("Example overlap label:", td_overlap_trees[0].label[0])


In [ ]:

def benchmark_generic_exact(trees, pairs, mode):
    if mode == "equality":
        w = w_eq
    elif mode == "overlap":
        w = w_overlap
    else:
        raise ValueError("mode must be 'equality' or 'overlap'")

    outputs = []
    t0 = time.perf_counter()
    for i, j in pairs:
        res = align_trees_algorithm1(trees[i], trees[j], w=w)
        outputs.append((res.path_internal, float(res.score)))
    sec = time.perf_counter() - t0
    return sec, outputs

def benchmark_fast_pairfit(trees, pairs, mode, token_weights):
    outputs = []
    t0 = time.perf_counter()
    for i, j in pairs:
        fm = FastTreePathMatcher(mode=mode, token_weights=token_weights)
        fm.fit(trees[i], trees[j])
        path, score = fm.predict()
        outputs.append((path, float(score)))
    sec = time.perf_counter() - t0
    return sec, outputs

def benchmark_fast_preencoded(trees, pairs, mode, token_weights):
    fm = FastTreePathMatcher(mode=mode, token_weights=token_weights)
    fm.fit_encoder(trees)
    encoded = [fm.encode_tree(T) for T in trees]

    outputs = []
    t0 = time.perf_counter()
    for i, j in pairs:
        path, score = fm.predict_encoded(encoded[i], encoded[j])
        outputs.append((path, float(score)))
    sec = time.perf_counter() - t0
    return sec, outputs

def summarize_results(mode, sec, outputs, baseline_outputs):
    score_mismatch = 0
    path_mismatch = 0
    for (path, score), (path0, score0) in zip(outputs, baseline_outputs):
        if abs(score - score0) > 1e-5:
            score_mismatch += 1
        if path != path0:
            path_mismatch += 1
    return {
        "mode": mode,
        "pairs": len(outputs),
        "sec": sec,
        "ms_per_pair": 1000.0 * sec / max(1, len(outputs)),
        "pairs_per_sec": len(outputs) / max(sec, 1e-12),
        "score_mismatch": score_mismatch,
        "path_mismatch": path_mismatch,
    }

# Warm up numba once so the benchmark reflects steady-state runtime.
if HAVE_NUMBA:
    _ = benchmark_fast_pairfit(td_eq_trees, pairs[:1], mode="equality", token_weights=token_weights)
    _ = benchmark_fast_pairfit(td_overlap_trees, pairs[:1], mode="overlap", token_weights=token_weights)
    print("Numba kernels warmed up.")


## Equality benchmark

In [ ]:

sec_base_eq, out_base_eq = benchmark_generic_exact(td_eq_trees, pairs, mode="equality")
sec_fast_eq_pair, out_fast_eq_pair = benchmark_fast_pairfit(td_eq_trees, pairs, mode="equality", token_weights=token_weights)
sec_fast_eq_pre, out_fast_eq_pre = benchmark_fast_preencoded(td_eq_trees, pairs, mode="equality", token_weights=token_weights)

eq_rows = []
eq_rows.append({"method": "generic_exact", **summarize_results("equality", sec_base_eq, out_base_eq, out_base_eq)})
eq_rows.append({"method": "fast_pairfit", **summarize_results("equality", sec_fast_eq_pair, out_fast_eq_pair, out_base_eq)})
eq_rows.append({"method": "fast_preencoded", **summarize_results("equality", sec_fast_eq_pre, out_fast_eq_pre, out_base_eq)})

eq_df = pd.DataFrame(eq_rows)
display(eq_df.round(4))


## Overlap benchmark

In [ ]:

sec_base_ov, out_base_ov = benchmark_generic_exact(td_overlap_trees, pairs, mode="overlap")
sec_fast_ov_pair, out_fast_ov_pair = benchmark_fast_pairfit(td_overlap_trees, pairs, mode="overlap", token_weights=token_weights)
sec_fast_ov_pre, out_fast_ov_pre = benchmark_fast_preencoded(td_overlap_trees, pairs, mode="overlap", token_weights=token_weights)

ov_rows = []
ov_rows.append({"method": "generic_exact", **summarize_results("overlap", sec_base_ov, out_base_ov, out_base_ov)})
ov_rows.append({"method": "fast_pairfit", **summarize_results("overlap", sec_fast_ov_pair, out_fast_ov_pair, out_base_ov)})
ov_rows.append({"method": "fast_preencoded", **summarize_results("overlap", sec_fast_ov_pre, out_fast_ov_pre, out_base_ov)})

ov_df = pd.DataFrame(ov_rows)
display(ov_df.round(4))


## Combined summary

In [ ]:

summary_df = pd.concat([eq_df, ov_df], ignore_index=True)
display(summary_df.round(4))

for mode_name, sub in summary_df.groupby("mode"):
    plt.figure(figsize=(6, 4))
    plt.bar(sub["method"], sub["ms_per_pair"])
    plt.ylabel("ms / pair")
    plt.title(f"{mode_name}: runtime per pair")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()


## Notes

- `fast_pairfit` still rebuilds the token encoder separately for each pair. It is useful when you only do a few matches.
- `fast_preencoded` is the intended high-throughput regime: fit one encoder on the full dataset, encode each tree once, then reuse those encodings for many pairwise matches.
- If your labels are structured objects, pass `label_getter=` to `FastTreePathMatcher` so the fast matcher only sees the specific token field you want to match on.
